[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Consultas diferidas y datos grandes.

Hasta ahora, cada operación de *Polars* se ha ejecutado en cuanto se escribe. Este *notebook* presenta el **modo diferido** (*lazy*), en el que *Polars* recibe la consulta completa, la optimiza y sólo entonces la ejecuta, y las herramientas para trabajar con datos más grandes que la memoria disponible:

1. Modo inmediato y modo diferido.
2. Lectura diferida de archivos.
3. El plan de consulta y sus optimizaciones.
4. La ejecución con `collect()` y el motor *streaming*.
5. Escritura por partes y conjuntos de datos particionados.
6. Un ejemplo completo con un archivo de ventas grande.

Al final hay cuatro secciones de autoestudio, marcadas con 📘: *SQL* en *Polars*, varias consultas a la vez, destinos particionados y lotes, y una medición de rendimiento entre *pandas* y *Polars*.

Este *notebook* se apoya en los *notebooks* 35 y 36 (*dataframes*, contextos y expresiones) y en la sección «Combinación de tablas» del 37.

* La siguiente celda importa las bibliotecas, configura la presentación de los números de punto flotante y crea el directorio `data_tmp` para los archivos que se generen.

In [ ]:
import os
import shutil
import time

import numpy as np
import polars as pl

pl.Config.set_fmt_float('full')
dir_tmp = 'data_tmp'
os.makedirs(dir_tmp, exist_ok=True)
print(f"Polars {pl.__version__}")

## Los datos de este *notebook*.

* Los ejemplos de la *API* usan los datos sintéticos de ventas descritos en el *notebook* 34: `data/ventas.csv` (20,000 ventas) y `data/tiendas.csv` (catálogo de tiendas).
* El ejemplo completo usa un archivo de ventas **grande**, que se genera con *NumPy* en `data_tmp/` al ejecutar el *notebook*; no se descarga nada.

El tamaño del archivo grande se controla con una **variable de entorno**: un valor con nombre que el sistema operativo entrega a los programas al iniciarlos. La función `os.environ.get(<nombre>, <valor_por_omisión>)` regresa el valor de la variable como texto, o el valor por omisión si la variable no existe.

| Variable | Valor por omisión | Uso |
| :--- | :--- | :--- |
| `PY311_RENGLONES` | `1000000` (un millón de renglones) | Número de ventas del archivo grande. La prueba automática del repositorio usa `20000` para terminar en pocos segundos. |

Con un millón de renglones, el archivo ocupa unos 63 MB en *CSV* y unos 12 MB en *Parquet*, se genera en uno o dos segundos en una máquina de *GitHub Codespaces* de 2 núcleos, y las diferencias entre el modo inmediato, el diferido y los archivos particionados ya se miden en décimas de segundo. Para probar con más datos, cambia el valor por omisión en la siguiente celda; cada millón de renglones adicional ocupa unos 100 MB más en disco, contando todos los archivos que crea el *notebook*.

* La siguiente celda lee la variable de entorno `PY311_RENGLONES` y la convierte en entero.

In [ ]:
N_RENGLONES = int(os.environ.get('PY311_RENGLONES', '1000000'))
print(f"Renglones del archivo grande: {N_RENGLONES:,}")

* La siguiente celda define las rutas de los datos de ejemplo y lee las ventas en modo inmediato, como en los *notebooks* 35 a 37.

In [ ]:
ruta_ventas = os.path.join('data', 'ventas.csv')
ruta_tiendas = os.path.join('data', 'tiendas.csv')

ventas = pl.read_csv(ruta_ventas, try_parse_dates=True)
ventas.shape

## Modo inmediato y modo diferido.

* En el **modo inmediato** (*eager*), que se ha usado hasta ahora, cada operación se ejecuta en cuanto se escribe y produce un `DataFrame`.
* En el **modo diferido** (*lazy*), las operaciones se acumulan en un **plan de consulta**: la descripción de los pasos que hay que hacer (leer, filtrar, calcular, agrupar...) sin ejecutarlos todavía. El plan se representa con un `pl.LazyFrame` y sólo se ejecuta al llamar a `collect()`.

Antes de ejecutar un plan, *Polars* lo revisa completo y lo optimiza. Por ejemplo, si al final sólo se usan dos columnas de un archivo, no lee las demás, y si hay un filtro, lo aplica mientras lee.

```
<df>.lazy()
<lf>.collect_schema()
```

Donde:

* `<df>` es un `DataFrame`. `lazy()` regresa un `LazyFrame` cuyo plan comienza con esos datos; no los copia.
* `<lf>` es un `LazyFrame`. `collect_schema()` regresa el *schema* (los nombres y tipos de las columnas) que tendrá el resultado, sin ejecutar la consulta.

Ninguno de los dos métodos recibe argumentos. Un `LazyFrame` admite casi todos los métodos de un `DataFrame` (`select()`, `with_columns()`, `filter()`, `group_by()`, `join()`, `sort()`, `head()`, ...), y cada uno regresa otro `LazyFrame` con un paso más en el plan.

| Método | Referencia |
| :--- | :--- |
| `df.lazy()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.lazy.html) |
| `lf.collect_schema()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect_schema.html) |

[Guía del modo diferido](https://docs.pola.rs/user-guide/lazy/)

**Ejemplos:**

* La siguiente celda convierte las ventas en un `LazyFrame` y le agrega dos pasos: un filtro de las ventas en línea y una columna con el importe bruto. El resultado es un `LazyFrame`: todavía no se ha calculado nada.

In [ ]:
plan = (
    ventas.lazy()
    .filter(pl.col('canal') == 'en línea')
    .with_columns(importe=pl.col('cantidad') * pl.col('precio_unitario'))
)
type(plan)

* La siguiente celda obtiene el *schema* del resultado sin ejecutar la consulta. La columna nueva, `importe`, ya aparece con su tipo.

In [ ]:
plan.collect_schema()

* La siguiente celda ejecuta el plan con `collect()`, que se estudia más adelante, y muestra los tres primeros renglones del `DataFrame` resultante.

In [ ]:
plan.collect().head(3)

## Lectura diferida: `scan_csv()` y `scan_parquet()`.

Las funciones `read_*()` leen el archivo completo en cuanto se llaman. Las funciones `scan_*()` crean un `LazyFrame` cuyo primer paso es leer el archivo: la lectura ocurre al ejecutar el plan, y sólo se lee lo que la consulta necesita.

```
pl.scan_csv(<ruta>, separator=<separador>, schema_overrides=<tipos>, try_parse_dates=<booleano>, n_rows=<número>, null_values=<nulos>, glob=<booleano>)
pl.scan_parquet(<ruta>, hive_partitioning=<booleano>, use_statistics=<booleano>, n_rows=<número>)
```

Donde:

* `<ruta>` es la ruta de un archivo, una lista de rutas o un patrón con comodines, como `'datos/parte_*.csv'`, donde `*` representa cualquier texto. Si la ruta corresponde a varios archivos, se leen como una sola tabla.
* `separator` es el carácter que separa los campos del *CSV*. Por omisión es `','`.
* `schema_overrides` es opcional: un diccionario `{columna: tipo}` que fija el tipo de algunas columnas; las demás se infieren.
* `try_parse_dates` indica si se intenta convertir en fecha, o en fecha y hora, las columnas de texto con ese formato. Por omisión es `False`.
* `n_rows` es opcional: el número máximo de renglones que se leen. Es útil para explorar un archivo grande.
* `null_values` es opcional: el texto, o la lista de textos, que se interpreta como nulo; por ejemplo, `'sin dato'`.
* `glob` indica si los caracteres `*` y `?` de la ruta se interpretan como comodines. Por omisión es `True`.
* `hive_partitioning` indica si los nombres de las carpetas con el esquema `columna=valor` (explicado más adelante en este *notebook*) se convierten en columnas. Por omisión es `None`: se activa al leer una carpeta y no al leer un patrón de archivos.
* `use_statistics` indica si se usan las estadísticas que *Parquet* guarda para cada grupo de renglones (*row group*, visto en el *notebook* 34), como el mínimo y el máximo de cada columna, para saltar los grupos que no pueden cumplir un filtro. Por omisión es `True`.

| Función | Referencia |
| :--- | :--- |
| `pl.scan_csv()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_csv.html) |
| `pl.scan_parquet()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html) |

Existen también `pl.scan_ipc()` (*Feather*) y `pl.scan_ndjson()` (*JSON* por renglones), con argumentos similares.

**Ejemplos:**

* La siguiente celda lee de forma diferida los primeros 1,000 renglones de `data/ventas.csv`, con la columna `tienda_id` como entero de 16 bits y `cantidad` como entero de 8 bits, y muestra el *schema* del resultado.

In [ ]:
muestra = pl.scan_csv(
    ruta_ventas,
    try_parse_dates=True,
    schema_overrides={'tienda_id': pl.Int16, 'cantidad': pl.Int8},
    n_rows=1_000,
).collect()

print(muestra.shape)
muestra.schema

* La siguiente celda escribe un *CSV* pequeño que usa punto y coma como separador y el texto `sin dato` para los valores faltantes, y lo lee con `separator` y `null_values`.

In [ ]:
ruta_inventario = os.path.join(dir_tmp, 'inventario_38.csv')
with open(ruta_inventario, 'w', encoding='utf-8') as archivo:
    archivo.write('clave;piezas\nA-01;12\nA-02;sin dato\nA-03;7\n')

pl.scan_csv(ruta_inventario, separator=';', null_values='sin dato').collect()

* La siguiente celda guarda las ventas en *Parquet* con `write_parquet()` (visto en el *notebook* 35) y las lee de forma diferida con `scan_parquet()`, pidiendo sólo cinco renglones.

In [ ]:
ruta_ventas_parquet = os.path.join(dir_tmp, 'ventas_38.parquet')
ventas.write_parquet(ruta_ventas_parquet)

pl.scan_parquet(ruta_ventas_parquet, n_rows=5).collect()

## El plan de consulta: `explain()`.

```
<lf>.explain(format=<formato>, optimized=<booleano>, engine=<motor>, optimizations=<optimizaciones>)
```

Donde:

* `format` es la presentación del plan: `'plain'` (por omisión), texto con sangrías, o `'tree'`, un diagrama de cajas.
* `optimized` indica si se muestra el plan optimizado (`True`, por omisión) o tal como se escribió (`False`).
* `engine` es el motor para el que se muestra el plan. Acepta los mismos valores que `collect()`, que se explican en la siguiente sección; por omisión es `'auto'`.
* `optimizations` es opcional: un objeto `pl.QueryOptFlags` que activa o desactiva optimizaciones individuales. No se usa en este curso.

El plan en texto se lee **de abajo hacia arriba**: las líneas con más sangría son los primeros pasos (normalmente, la lectura de los archivos) y la primera línea es el último. En el diagrama de cajas, los primeros pasos son las cajas de abajo. Algunas palabras frecuentes en el plan son:

| En el plan | Significado |
| :--- | :--- |
| `Csv SCAN`, `Parquet SCAN`, `DF` | Lectura de un *CSV*, de un *Parquet* o de un `DataFrame` en memoria. |
| `PROJECT 3/10 COLUMNS` | Durante la lectura, sólo se leen 3 de las 10 columnas. |
| `SELECTION: ...` | Durante la lectura, sólo se conservan los renglones que cumplen la condición. |
| `FILTER`, `AGGREGATE`, `JOIN`, `SORT`, `SLICE` | Filtro, agrupación, combinación, ordenamiento y selección de renglones por posición. |
| `π` (letra griega *pi*) | Selección de columnas. |

### Optimizaciones.

| Optimización | Efecto |
| :--- | :--- |
| *Predicate pushdown* | Los filtros se aplican lo antes posible; si se lee un archivo, durante la lectura. |
| *Projection pushdown* | Sólo se leen y procesan las columnas que se usan en la consulta. |
| *Slice pushdown* | Si sólo se piden algunos renglones (`head()`), se deja de leer al obtenerlos. |
| Eliminación de subexpresiones comunes | Un cálculo que aparece varias veces se hace una sola vez. |

[Documentación de `explain()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.explain.html)

**Ejemplos:**

* La siguiente celda define una consulta diferida sobre los dos archivos *CSV*: el importe neto de las ventas en línea por región en 2025. Ninguna operación se ejecuta todavía.

In [ ]:
consulta = (
    pl.scan_csv(ruta_ventas, try_parse_dates=True)
    .join(pl.scan_csv(ruta_tiendas), on='tienda_id', how='inner')
    .with_columns(
        importe_neto=pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0))
    )
    .filter(pl.col('canal') == 'en línea', pl.col('fecha_hora').dt.year() == 2025)
    .group_by('region')
    .agg(pl.col('importe_neto').sum().round(2).alias('importe'))
    .sort('importe', descending=True)
)
type(consulta)

* La siguiente celda muestra el plan tal como se escribió. Leído de abajo hacia arriba: primero se leen los dos archivos completos, después se combinan y el filtro se aplica al final, sobre el resultado del *join*.

In [ ]:
print(consulta.explain(optimized=False))

* La siguiente celda muestra el plan optimizado. En la lectura del *CSV* de ventas (`Csv SCAN`) hay dos cambios: `PROJECT` indica que sólo se leen las columnas necesarias, y `SELECTION` que el filtro se aplica durante la lectura, antes del *join*.

In [ ]:
print(consulta.explain())

* La siguiente celda muestra, con `format='tree'`, el plan de una consulta más corta. Como sólo se piden cinco renglones con `head()`, la selección de renglones (`SLICE`) aparece también dentro de la lectura: *Polars* deja de leer el archivo en cuanto los obtiene.

In [ ]:
print(
    pl.scan_csv(ruta_ventas)
    .filter(pl.col('canal') == 'en línea')
    .select('categoria', 'cantidad')
    .head(5)
    .explain(format='tree')
)

## Ejecución: `collect()` y el motor *streaming*.

```
<lf>.collect(engine=<motor>, background=<booleano>, optimizations=<optimizaciones>)
```

Donde:

* `engine` es el motor que ejecuta la consulta:
  * `'in-memory'`: cada paso carga en memoria todos los datos que necesita.
  * `'streaming'`: procesa los datos por bloques de renglones, de modo que puede trabajar con datos más grandes que la memoria disponible; además, suele ser más rápido.
  * `'auto'` (por omisión): usa el motor configurado con `pl.Config.set_engine_affinity()` o con la variable de entorno `POLARS_ENGINE_AFFINITY`. **Si no hay ninguno configurado, en *Polars* 2.x usa el motor *streaming*.** Si el motor elegido no puede ejecutar alguna parte de la consulta, *Polars* usa el motor en memoria para ella.
  * `'gpu'` usa una tarjeta gráfica de *NVIDIA* y requiere una biblioteca adicional; no se usa en este curso.
* `background` indica si la consulta se ejecuta en segundo plano. Por omisión es `False`; con `True`, `collect()` regresa de inmediato un objeto con el que se obtiene el resultado después. No se usa en este curso.
* `optimizations` es el mismo argumento de `explain()`.

> **Nota sobre versiones.** En *Polars* 1.x, `collect()` usaba por omisión el motor en memoria. El material escrito para esa versión puede decir que `collect()` carga en memoria todos los datos de la consulta; con la versión fijada en este curso (2.1.0) ya no es así.

Los métodos `sink_*()`, que se estudian en la siguiente sección, siempre usan el motor *streaming*.

[Documentación de `collect()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect.html) · [Guía del motor *streaming*](https://docs.pola.rs/user-guide/concepts/streaming/)

**Ejemplo:**

* La siguiente celda ejecuta la consulta con el motor por omisión y comprueba que los motores en memoria y *streaming* producen el mismo resultado.

In [ ]:
resultado = consulta.collect()
print(resultado.equals(consulta.collect(engine='in-memory')))
print(resultado.equals(consulta.collect(engine='streaming')))
resultado

## Datos más grandes que la memoria: escritura por partes y particiones.

`collect()` reúne el resultado completo en memoria. Cuando el resultado es muy grande, o cuando sólo se quiere guardar para otro proceso, conviene usar `sink_*()`: ejecuta el plan con el motor *streaming* y escribe el resultado en un archivo conforme se calcula, sin reunirlo nunca completo en memoria.

```
<lf>.sink_parquet(<ruta>, compression=<algoritmo>, row_group_size=<renglones>, mkdir=<booleano>)
<lf>.sink_csv(<ruta>, separator=<separador>, include_header=<booleano>, mkdir=<booleano>)
<df>.write_parquet(<ruta>, partition_by=<columnas>)
```

Donde:

* `<ruta>` es el archivo que se escribe. En `write_parquet()` con `partition_by`, es la carpeta raíz del conjunto de datos.
* `compression` es el algoritmo de compresión de *Parquet*: `'zstd'` (por omisión), `'lz4'`, `'snappy'`, `'gzip'`, `'brotli'` o `'uncompressed'`.
* `row_group_size` es opcional: el número de renglones de cada grupo de renglones del archivo *Parquet*. Grupos más pequeños permiten saltar más datos con las estadísticas, a cambio de un archivo algo más grande.
* `mkdir` indica si se crean las carpetas de la ruta que no existan. Por omisión es `False`.
* `separator` es el carácter que separa los campos del *CSV* (por omisión `','`), e `include_header` indica si se escribe el renglón de encabezados (por omisión `True`).
* `partition_by` es la columna, o la lista de columnas, por la que se divide el conjunto de datos: cada valor distinto genera una carpeta. Como `write_parquet()` es un método de `DataFrame`, los datos deben estar en memoria.

| Método | Referencia |
| :--- | :--- |
| `lf.sink_parquet()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/api/polars.LazyFrame.sink_parquet.html) |
| `lf.sink_csv()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/api/polars.LazyFrame.sink_csv.html) |
| `df.write_parquet()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_parquet.html) |

### Conjuntos de datos particionados y el esquema de *Hive*.

Cuando los datos crecen, es común dividirlos en varios archivos organizados en carpetas según el valor de una o más columnas, por ejemplo una carpeta por año. A esto se le llama **particionar**. La forma más usada de nombrar las carpetas es el **esquema de *Hive*** (llamado así por *Apache Hive*, el sistema que lo popularizó): cada carpeta se llama `columna=valor`.

```
ventas_por_anio/
├── anio=2024/
│   └── 00000000.parquet
└── anio=2025/
    └── 00000000.parquet
```

Este esquema tiene dos ventajas:

* La columna de partición no se guarda dentro de los archivos: se reconstruye a partir del nombre de la carpeta.
* Una consulta que filtra por esa columna sólo abre las carpetas necesarias. `pl.scan_parquet(<carpeta>, hive_partitioning=True)` lee el conjunto completo como una sola tabla y aplica esta optimización.

**Ejemplos:**

* La siguiente celda convierte el *CSV* de ventas a *Parquet* con `sink_parquet()`, agregando el importe neto, sin cargar el archivo completo en memoria.

In [ ]:
ruta_neto = os.path.join(dir_tmp, 'ventas_neto_38.parquet')

(
    pl.scan_csv(ruta_ventas, try_parse_dates=True)
    .with_columns(
        importe_neto=(pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0))).round(2)
    )
    .sink_parquet(ruta_neto, compression='zstd')
)
print(f"{os.path.getsize(ruta_neto) / 1024:.0f} KB")

* La siguiente celda escribe las ventas en línea de 2025 en un *CSV* separado por punto y coma, dentro de una carpeta que todavía no existe; `mkdir=True` la crea.

In [ ]:
ruta_en_linea = os.path.join(dir_tmp, 'exportaciones_38', 'en_linea_2025.csv')

(
    pl.scan_csv(ruta_ventas, try_parse_dates=True)
    .filter(pl.col('canal') == 'en línea', pl.col('fecha_hora').dt.year() == 2025)
    .select('venta_id', 'fecha_hora', 'producto', 'cantidad')
    .sink_csv(ruta_en_linea, separator=';', mkdir=True)
)
pl.read_csv(ruta_en_linea, separator=';').shape

* La siguiente celda escribe las ventas particionadas por año con `write_parquet(partition_by=...)` y muestra los archivos creados con `os.walk()`, que recorre una carpeta y todas sus subcarpetas y regresa, para cada una, su ruta, sus subcarpetas y sus archivos. Antes borra la carpeta con `shutil.rmtree()` por si quedó de una ejecución anterior; `ignore_errors=True` evita un error si no existe.

In [ ]:
ruta_particiones = os.path.join(dir_tmp, 'ventas_por_anio_38')
shutil.rmtree(ruta_particiones, ignore_errors=True)

ventas.with_columns(anio=pl.col('fecha_hora').dt.year()).write_parquet(ruta_particiones, partition_by='anio')

for carpeta, _, archivos in sorted(os.walk(ruta_particiones)):
    if archivos:
        print(carpeta, archivos)

* La siguiente celda consulta sólo 2025. En el plan optimizado, la lectura (`Parquet SCAN`) sólo incluye el archivo de la carpeta `anio=2025`.

In [ ]:
por_anio = (
    pl.scan_parquet(ruta_particiones, hive_partitioning=True)
    .filter(pl.col('anio') == 2025)
    .select(
        pl.len().alias('ventas_2025'),
        (pl.col('cantidad') * pl.col('precio_unitario')).sum().round(2).alias('importe_bruto'),
    )
)
print(por_anio.explain())
por_anio.collect()

## Ejemplo completo: un archivo de ventas grande.

Este ejemplo reúne todo lo anterior con un archivo de `N_RENGLONES` ventas:

1. Se genera con *NumPy* en diez archivos *CSV*, como si cada uno llegara por separado desde otro sistema.
2. Se compara una consulta en modo inmediato y en modo diferido.
3. Se convierte a *Parquet* particionado por año sin cargarlo completo en memoria.
4. Se consulta un año y se comprueba en el plan que sólo se lee su carpeta.

Los datos tienen las mismas columnas principales que `data/ventas.csv`, para tres años completos (2023 a 2025) y doce tiendas, sin producto ni cliente. Los tiempos que se muestran dependen del equipo y del tamaño elegido.

* La siguiente celda define una función que genera con *NumPy* (*notebook* 04) un *dataframe* de `n` ventas sintéticas. Las fechas se obtienen sumando a la fecha inicial un número aleatorio de segundos, y el 60 % de los descuentos son nulos; `nan_to_null=True` convierte los `NaN` de *NumPy* en nulos de *Polars*.

In [ ]:
CATEGORIAS = np.array(['Deportes', 'Electrónica', 'Hogar', 'Papelería'])
CANALES = np.array(['tienda', 'en línea'])
INICIO = np.datetime64('2023-01-01T00:00:00', 's')
SEGUNDOS = int((np.datetime64('2026-01-01T00:00:00', 's') - INICIO) / np.timedelta64(1, 's'))


def generar_parte(rng, primer_id, n):
    """Regresa un dataframe con n ventas sintéticas; los identificadores empiezan en primer_id."""
    segundos = rng.integers(0, SEGUNDOS, n).astype('timedelta64[s]')
    descuento = rng.choice([0.05, 0.10, 0.15], n)
    return pl.DataFrame(
        {
            'venta_id': np.arange(primer_id, primer_id + n),
            'fecha_hora': (INICIO + segundos).astype('datetime64[us]'),
            'tienda_id': rng.integers(1, 13, n),
            'canal': rng.choice(CANALES, n, p=[0.7, 0.3]),
            'categoria': rng.choice(CATEGORIAS, n),
            'cantidad': rng.integers(1, 6, n),
            'precio_unitario': rng.uniform(20, 2000, n).round(2),
            'descuento': np.where(rng.random(n) < 0.6, np.nan, descuento),
        },
        nan_to_null=True,
    )

* La siguiente celda genera las ventas en diez partes y escribe cada una en un *CSV* de `data_tmp/ventas_grandes_38/`. Como cada parte se escribe antes de generar la siguiente, en memoria sólo hay una décima parte de los datos a la vez. La semilla fija (`38`) hace que los datos sean siempre los mismos.

In [ ]:
dir_grandes = os.path.join(dir_tmp, 'ventas_grandes_38')
shutil.rmtree(dir_grandes, ignore_errors=True)
os.makedirs(dir_grandes)

rng = np.random.default_rng(38)
partes = 10
por_parte = N_RENGLONES // partes

inicio = time.perf_counter()
for numero in range(partes):
    n = por_parte if numero < partes - 1 else N_RENGLONES - por_parte * (partes - 1)
    parte = generar_parte(rng, numero * por_parte + 1, n)
    parte.write_csv(os.path.join(dir_grandes, f'parte_{numero:02d}.csv'))
segundos = time.perf_counter() - inicio

tamano_csv = sum(os.path.getsize(os.path.join(dir_grandes, nombre)) for nombre in os.listdir(dir_grandes))
print(f"{N_RENGLONES:,} renglones en {partes} archivos · {tamano_csv / 1024**2:.1f} MB · {segundos:.1f} s")

* La siguiente celda define una función que ejecuta otra función varias veces y regresa el menor de los tiempos. Se toma el menor porque la primera ejecución puede ser más lenta, por ejemplo porque el sistema operativo todavía no tiene el archivo en su memoria caché.

In [ ]:
def mejor_tiempo(funcion, repeticiones=3):
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - inicio)
    return min(tiempos)

* La siguiente celda calcula las piezas vendidas en línea por categoría de dos maneras, leyendo los diez archivos con el patrón `parte_*.csv`: en modo inmediato, con `read_csv()`, que lee todas las columnas y renglones antes de filtrar; y en modo diferido, con `scan_csv()`, que sólo lee tres columnas y aplica el filtro durante la lectura. Primero comprueba que ambos resultados coinciden y después mide los tiempos.

In [ ]:
patron_csv = os.path.join(dir_grandes, 'parte_*.csv')


def inmediato():
    datos = pl.read_csv(patron_csv, try_parse_dates=True)
    return datos.filter(pl.col('canal') == 'en línea').group_by('categoria').agg(pl.col('cantidad').sum()).sort('categoria')


def diferido():
    return (
        pl.scan_csv(patron_csv, try_parse_dates=True)
        .filter(pl.col('canal') == 'en línea')
        .group_by('categoria')
        .agg(pl.col('cantidad').sum())
        .sort('categoria')
        .collect()
    )


print(f"Mismo resultado: {inmediato().equals(diferido())}")
t_inmediato = mejor_tiempo(inmediato)
t_diferido = mejor_tiempo(diferido)
print(f"Inmediato: {t_inmediato:.3f} s")
print(f"Diferido:  {t_diferido:.3f} s")

* La siguiente celda convierte los diez *CSV* a un conjunto de datos *Parquet* particionado por año, **sin cargarlos completos en memoria**: obtiene la lista de años con una consulta diferida pequeña y, para cada año, una consulta diferida filtra sus ventas y `sink_parquet()` las escribe en la carpeta `anio=<año>`. La columna `anio` se elimina antes de escribir, porque el esquema de *Hive* la guarda en el nombre de la carpeta.

In [ ]:
dir_particionado = os.path.join(dir_tmp, 'ventas_grandes_por_anio_38')
shutil.rmtree(dir_particionado, ignore_errors=True)

ventas_grandes = pl.scan_csv(patron_csv, try_parse_dates=True).with_columns(anio=pl.col('fecha_hora').dt.year())
anios = ventas_grandes.select(pl.col('anio').unique().sort()).collect()['anio'].to_list()

inicio = time.perf_counter()
for anio in anios:
    (
        ventas_grandes
        .filter(pl.col('anio') == anio)
        .drop('anio')
        .sink_parquet(os.path.join(dir_particionado, f'anio={anio}', 'datos.parquet'), mkdir=True)
    )
print(f"Años: {anios} · {time.perf_counter() - inicio:.1f} s")

for carpeta, _, archivos in sorted(os.walk(dir_particionado)):
    if archivos:
        print(carpeta, archivos)

* La siguiente celda define una consulta sobre el conjunto particionado: el número de ventas y el importe bruto de 2025 por categoría. En el plan, la lectura (`Parquet SCAN`) sólo incluye el archivo de `anio=2025`, y `PROJECT` indica que sólo se leen las columnas necesarias.

In [ ]:
consulta_2025 = (
    pl.scan_parquet(dir_particionado, hive_partitioning=True)
    .filter(pl.col('anio') == 2025)
    .group_by('categoria')
    .agg(
        pl.len().alias('ventas'),
        (pl.col('cantidad') * pl.col('precio_unitario')).sum().round(2).alias('importe_bruto'),
    )
    .sort('categoria')
)
print(consulta_2025.explain())
consulta_2025.collect()

* La siguiente celda hace la misma consulta directamente sobre los *CSV*, comprueba que el número de ventas por categoría coincide y compara los tiempos y el espacio en disco de ambos formatos.

In [ ]:
def desde_csv():
    return (
        pl.scan_csv(patron_csv, try_parse_dates=True)
        .filter(pl.col('fecha_hora').dt.year() == 2025)
        .group_by('categoria')
        .agg(
            pl.len().alias('ventas'),
            (pl.col('cantidad') * pl.col('precio_unitario')).sum().round(2).alias('importe_bruto'),
        )
        .sort('categoria')
        .collect()
    )


print(f"Mismas ventas por categoría: {desde_csv().select('categoria', 'ventas').equals(consulta_2025.collect().select('categoria', 'ventas'))}")

t_csv = mejor_tiempo(desde_csv)
t_parquet = mejor_tiempo(consulta_2025.collect)
tamano_parquet = sum(
    os.path.getsize(os.path.join(carpeta, nombre)) for carpeta, _, archivos in os.walk(dir_particionado) for nombre in archivos
)
print(f"CSV:                   {t_csv:.3f} s · {tamano_csv / 1024**2:6.1f} MB")
print(f"Parquet particionado:  {t_parquet:.3f} s · {tamano_parquet / 1024**2:6.1f} MB")

La consulta sobre el conjunto particionado es mucho más rápida por tres razones que se suman: sólo abre la carpeta de un año, sólo lee las columnas que usa y no tiene que interpretar texto, porque *Parquet* guarda los datos con su tipo. Además, los archivos ocupan varias veces menos espacio que los *CSV*.

Todo el proceso (generación, conversión y consultas) se hizo sin tener nunca el archivo completo en memoria: el mismo código funciona con archivos más grandes que la memoria disponible, siempre que quepan en el disco.

## ¿Inmediato o diferido? ¿`collect()` o `sink_*()`?

| Situación | Recomendación |
| :--- | :--- |
| Explorar datos pequeños paso a paso en un *notebook*. | Modo inmediato: `read_*()` y métodos de `DataFrame`. |
| Un proceso con varios pasos, o archivos medianos o grandes. | Modo diferido: `scan_*()` y `collect()`. Una práctica común es explorar en modo inmediato con una muestra (`n_rows`) y escribir el proceso final en modo diferido. |
| El resultado cabe en memoria. | `collect()`. |
| El resultado no cabe en memoria, o sólo se guarda para otro proceso. | `sink_parquet()` o `sink_csv()`. |
| Las consultas filtran casi siempre por una misma columna (fecha, región, ...). | Particionar por esa columna con el esquema de *Hive* y leer con `scan_parquet()`. |
| Los datos no caben en el disco de una sola máquina, o el proceso necesita varias máquinas. | Un sistema de procesamiento distribuido, como *Apache Spark* (curso py331). Queda fuera del alcance de este curso. |

## 📘 Autoestudio: *SQL* en *Polars*.

> **📘 Autoestudio.** Esta sección no se ve en clase y ningún otro *notebook* ni sección de clase depende de ella.
>
> * **Qué aprenderás:** consultar *dataframes* y *lazyframes* con *SQL*, una o varias tablas a la vez.
> * **Requisitos:** la sección «Combinación de tablas» del *notebook* 37 y las secciones «Modo inmediato y modo diferido» y «Ejemplo completo» de este *notebook*. Se supone que conoces la sintaxis básica de `SELECT`, `WHERE`, `GROUP BY` y `JOIN`.
> * **Tiempo estimado:** 15 minutos.
> * **Cómo saber si lo hiciste bien:** al final hay un ejercicio y una celda que comprueba tu resultado.

*Polars* puede ejecutar consultas *SQL* sobre *dataframes* y *lazyframes*. Las consultas se traducen al mismo plan que las expresiones, así que se optimizan igual. Es útil para equipos que ya conocen *SQL* o para migrar consultas existentes; el dialecto es similar al de *PostgreSQL*.

```
<df>.sql(<consulta>, table_name=<nombre>)
<lf>.sql(<consulta>, table_name=<nombre>)
pl.SQLContext(<tabla_1>=<frame_1>, <tabla_2>=<frame_2>, ..., eager=<booleano>)
<contexto>.execute(<consulta>, eager=<booleano>)
```

Donde:

* `<consulta>` es el texto de la consulta *SQL*.
* `table_name` es el nombre con el que la consulta hace referencia al *dataframe* o *lazyframe*. Por omisión es `'self'`. Con un `DataFrame` el resultado es un `DataFrame`; con un `LazyFrame`, un `LazyFrame`.
* `<tabla_i>=<frame_i>` registra un `DataFrame` o un `LazyFrame` con un nombre de tabla, para combinar varias tablas en una consulta.
* `eager`, en `pl.SQLContext()`, indica qué regresa `execute()` por omisión: un `LazyFrame` (`False`, por omisión) o un `DataFrame` (`True`). En `execute()`, el mismo argumento cambia el comportamiento sólo para esa consulta.
* `<contexto>` es el objeto que regresa `pl.SQLContext()`.

| Función o método | Referencia |
| :--- | :--- |
| `df.sql()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sql.html) |
| `lf.sql()` | [Documentación](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.sql.html) |
| `pl.SQLContext` | [Documentación](https://docs.pola.rs/api/python/stable/reference/sql/python_api.html) |

[Guía de *SQL*](https://docs.pola.rs/user-guide/sql/intro/)

**Ejemplos:**

* La siguiente celda calcula con `df.sql()` el número de ventas y el ticket promedio por canal. `COALESCE(descuento, 0)` sustituye los descuentos nulos por cero, como `fill_null(0)`.

In [ ]:
ventas.sql('''
    SELECT canal,
           COUNT(*) AS ventas,
           ROUND(AVG(cantidad * precio_unitario * (1 - COALESCE(descuento, 0))), 2) AS ticket_promedio
    FROM self
    GROUP BY canal
    ORDER BY ventas DESC
''')

* La siguiente celda usa `lf.sql()` sobre el conjunto particionado del ejemplo completo. El resultado es un `LazyFrame`, así que se ejecuta con `collect()`.

In [ ]:
pl.scan_parquet(dir_particionado, hive_partitioning=True).sql('''
    SELECT anio, canal, COUNT(*) AS ventas
    FROM self
    GROUP BY anio, canal
    ORDER BY anio, canal
''').collect()

* La siguiente celda registra las ventas y el catálogo de tiendas en un `SQLContext` y obtiene las tres tiendas con mayor importe bruto en 2025, combinando ambas tablas.

In [ ]:
contexto = pl.SQLContext(ventas=ventas.lazy(), tiendas=pl.scan_csv(ruta_tiendas))

contexto.execute('''
    SELECT t.nombre, ROUND(SUM(v.cantidad * v.precio_unitario), 2) AS importe_bruto
    FROM ventas AS v
    INNER JOIN tiendas AS t ON v.tienda_id = t.tienda_id
    WHERE EXTRACT(YEAR FROM v.fecha_hora) = 2025
    GROUP BY t.nombre
    ORDER BY importe_bruto DESC
    LIMIT 3
''').collect()

### Ejercicio.

Con *SQL* sobre `pl.scan_parquet(dir_particionado, hive_partitioning=True)`, obtén `por_canal_sql`: un `DataFrame` con un renglón por canal para las ventas de **2025** y tres columnas, `canal`, `ventas` (número de ventas) y `piezas` (suma de `cantidad`), ordenado por `canal`.

* La siguiente celda es para tu solución: sustituye `None` por el código que calcula `por_canal_sql`.

In [ ]:
por_canal_sql = None  # Sustituye None por tu solución.

**Comprueba tu resultado.**

* La siguiente celda comprueba tu resultado con `assert`. Si algo no coincide, muestra un mensaje que indica qué revisar; si todavía no resuelves el ejercicio, sólo lo indica.

In [ ]:
def comprobar_por_canal(resultado):
    assert not isinstance(resultado, pl.LazyFrame), 'El resultado es un LazyFrame: ejecútalo con collect().'
    assert isinstance(resultado, pl.DataFrame), 'El resultado debe ser un DataFrame de Polars.'
    assert resultado.columns == ['canal', 'ventas', 'piezas'], f'Las columnas deben ser canal, ventas y piezas; se obtuvo {resultado.columns}.'
    esperado = (
        pl.scan_parquet(dir_particionado, hive_partitioning=True)
        .filter(pl.col('anio') == 2025)
        .group_by('canal')
        .agg(pl.len().cast(pl.Int64).alias('ventas'), pl.col('cantidad').sum().cast(pl.Int64).alias('piezas'))
        .sort('canal')
        .collect()
    )
    obtenido = resultado.with_columns(pl.col('ventas', 'piezas').cast(pl.Int64))
    assert obtenido.equals(esperado), 'Los valores no coinciden; revisa el filtro del año, la agrupación y el orden.'
    print('Correcto.')


if por_canal_sql is None:
    print('Ejercicio pendiente: completa la celda anterior y vuelve a ejecutar esta.')
else:
    comprobar_por_canal(por_canal_sql)

**Solución.**

* La siguiente celda contiene una solución posible y la comprueba con la misma función. Su código está oculto: intenta resolver el ejercicio antes de mostrarlo (haz clic en la barra a la izquierda de la celda).

In [ ]:
por_canal_sql = pl.scan_parquet(dir_particionado, hive_partitioning=True).sql('''
    SELECT canal, COUNT(*) AS ventas, SUM(cantidad) AS piezas
    FROM self
    WHERE anio = 2025
    GROUP BY canal
    ORDER BY canal
''').collect()

comprobar_por_canal(por_canal_sql)

## 📘 Autoestudio: Varias consultas a la vez.

> **📘 Autoestudio.** Esta sección no se ve en clase y ningún otro *notebook* ni sección de clase depende de ella.
>
> * **Qué aprenderás:** ejecutar varias consultas diferidas juntas para que compartan el trabajo común.
> * **Requisitos:** las secciones «El plan de consulta» y «Ejecución» de este *notebook*.
> * **Tiempo estimado:** 5 minutos.
> * **Cómo saber si lo hiciste bien:** al final hay un ejercicio y una celda que comprueba tu resultado.

Cuando varias consultas parten de los mismos datos, ejecutarlas una por una repite el trabajo común, como leer el mismo archivo. [`pl.collect_all()`](https://docs.pola.rs/api/python/stable/reference/api/polars.collect_all.html) recibe una lista de consultas, las optimiza juntas (las partes comunes se calculan una sola vez) y las ejecuta en paralelo.

```
pl.collect_all(<consultas>, engine=<motor>, optimizations=<optimizaciones>, lazy=<booleano>)
```

Donde:

* `<consultas>` es una lista de `LazyFrame`. El resultado es una lista de `DataFrame`, en el mismo orden.
* `engine` es el motor, con los mismos valores que en `collect()`.
* `optimizations` es el mismo argumento de `explain()` y `collect()`.
* `lazy` indica si, en lugar de ejecutar, se regresa un `LazyFrame` para ejecutarlo después. Por omisión es `False`; sólo es válido cuando todas las consultas escriben a disco con `sink_*()`, y no se usa en este curso.

**Ejemplo:**

* La siguiente celda define dos consultas sobre los mismos archivos *CSV* grandes y compara el tiempo de ejecutarlas por separado con el de ejecutarlas juntas.

In [ ]:
base = pl.scan_csv(patron_csv).filter(pl.col('canal') == 'en línea')
por_categoria = base.group_by('categoria').agg(pl.len().alias('ventas')).sort('categoria')
por_tienda = base.group_by('tienda_id').agg(pl.col('cantidad').sum().alias('piezas')).sort('tienda_id')

t_separadas = mejor_tiempo(lambda: [por_categoria.collect(), por_tienda.collect()])
t_juntas = mejor_tiempo(lambda: pl.collect_all([por_categoria, por_tienda]))
print(f"Por separado: {t_separadas:.3f} s · Juntas: {t_juntas:.3f} s")

resultado_categoria, resultado_tienda = pl.collect_all([por_categoria, por_tienda])
resultado_categoria

### Ejercicio.

Más abajo se define `consultas_ejercicio`, una lista con tres consultas sobre el conjunto particionado. Obtén `resultados_juntos`: la lista de resultados de las tres, ejecutadas **juntas** con una sola llamada.

* La siguiente celda define las tres consultas del ejercicio: ventas por año, piezas por canal y precio unitario promedio por categoría.

In [ ]:
particionado = pl.scan_parquet(dir_particionado, hive_partitioning=True)
consultas_ejercicio = [
    particionado.group_by('anio').agg(pl.len().alias('ventas')).sort('anio'),
    particionado.group_by('canal').agg(pl.col('cantidad').sum().alias('piezas')).sort('canal'),
    particionado.group_by('categoria').agg(pl.col('precio_unitario').mean().round(2).alias('precio_medio')).sort('categoria'),
]

* La siguiente celda es para tu solución: sustituye `None` por el código que calcula `resultados_juntos`.

In [ ]:
resultados_juntos = None  # Sustituye None por tu solución.

**Comprueba tu resultado.**

* La siguiente celda comprueba tu resultado con `assert`. Si algo no coincide, muestra un mensaje que indica qué revisar; si todavía no resuelves el ejercicio, sólo lo indica.

In [ ]:
def comprobar_juntos(resultado):
    assert isinstance(resultado, list) and len(resultado) == 3, 'El resultado debe ser una lista con tres DataFrame.'
    for posicion, (obtenido, consulta_i) in enumerate(zip(resultado, consultas_ejercicio)):
        assert isinstance(obtenido, pl.DataFrame), f'El elemento {posicion} no es un DataFrame.'
        assert obtenido.equals(consulta_i.collect()), f'El resultado {posicion} no coincide con su consulta.'
    print('Correcto.')


if resultados_juntos is None:
    print('Ejercicio pendiente: completa la celda anterior y vuelve a ejecutar esta.')
else:
    comprobar_juntos(resultados_juntos)

**Solución.**

* La siguiente celda contiene una solución posible y la comprueba con la misma función. Su código está oculto: intenta resolver el ejercicio antes de mostrarlo (haz clic en la barra a la izquierda de la celda).

In [ ]:
resultados_juntos = pl.collect_all(consultas_ejercicio)

comprobar_juntos(resultados_juntos)

## 📘 Autoestudio: Destinos particionados y lotes.

> **📘 Autoestudio.** Esta sección no se ve en clase y ningún otro *notebook* ni sección de clase depende de ella.
>
> * **Qué aprenderás:** escribir un conjunto particionado en una sola pasada y recorrer el resultado de una consulta por lotes.
> * **Requisitos:** la sección «Datos más grandes que la memoria» y el ejemplo completo de este *notebook*.
> * **Tiempo estimado:** 10 minutos.
> * **Cómo saber si lo hiciste bien:** al final hay un ejercicio y una celda que comprueba tu resultado.

En el ejemplo completo, el conjunto particionado se escribió con un ciclo: una consulta por año, lo que lee los *CSV* una vez por cada año. [`pl.PartitionBy`](https://docs.pola.rs/api/python/stable/reference/api/polars.PartitionBy.html) indica a `sink_parquet()` (o a `sink_csv()`) que divida el resultado en carpetas **en una sola pasada**. Y cuando cada parte del resultado debe procesarse con código propio (por ejemplo, para enviarla a otro sistema), [`collect_batches()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect_batches.html) entrega el resultado por lotes de renglones, sin reunirlo completo en memoria.

```
<lf>.sink_parquet(pl.PartitionBy(<carpeta>, key=<columnas>, include_key=<booleano>, max_rows_per_file=<renglones>, approximate_bytes_per_file=<bytes>, file_path_provider=<función>), mkdir=True)
<lf>.collect_batches(chunk_size=<renglones>, maintain_order=<booleano>, lazy=<booleano>, engine=<motor>, optimizations=<optimizaciones>)
```

Donde, en `pl.PartitionBy`:

* `<carpeta>` es la carpeta raíz del conjunto de datos.
* `key` es la columna, o la lista de columnas o expresiones, por la que se divide; las carpetas siguen el esquema de *Hive* (`columna=valor`). Los caracteres que no son *ASCII*, como las letras acentuadas, se codifican en el nombre de la carpeta (por ejemplo, `ó` se escribe `%C3%B3`), y `scan_parquet()` los decodifica al leer.
* `include_key` indica si la columna de partición también se guarda dentro de los archivos. Por omisión, se guarda.
* `max_rows_per_file` es opcional: el número máximo de renglones de cada archivo; si una partición tiene más, se divide en varios archivos.
* `approximate_bytes_per_file` es el tamaño aproximado de cada archivo, medido según el tamaño de los datos en memoria. Por omisión es `'auto'`.
* `file_path_provider` es opcional: una función que decide el nombre de cada archivo. No se usa en este curso.

Y en `collect_batches()`:

* `chunk_size` es opcional: el número de renglones de cada lote. Si se omite, *Polars* lo decide.
* `maintain_order` indica si los lotes respetan el orden de los datos. Por omisión es `True`; con `False` es un poco más rápido.
* `lazy` indica si la consulta empieza a ejecutarse hasta que se pide el primer lote. Por omisión es `False`.
* `engine` y `optimizations` son los mismos argumentos de `collect()`.

> **Advertencia.** En *Polars* 2.1.0, ambas funciones están marcadas como **inestables**: su comportamiento puede cambiar en versiones futuras. Además, `collect_batches()` es más lenta que `sink_*()`; conviene usarla sólo cuando el proceso no puede expresarse con una escritura a archivo.

**Ejemplos:**

* La siguiente celda escribe el conjunto particionado por año en una sola pasada con `pl.PartitionBy` y muestra los archivos creados.

In [ ]:
dir_por_anio_pb = os.path.join(dir_tmp, 'ventas_grandes_por_anio_pb_38')
shutil.rmtree(dir_por_anio_pb, ignore_errors=True)

ventas_grandes.sink_parquet(pl.PartitionBy(dir_por_anio_pb, key='anio'), mkdir=True)

for carpeta, _, archivos in sorted(os.walk(dir_por_anio_pb)):
    if archivos:
        print(carpeta, archivos)

* La siguiente celda recorre por lotes de 100,000 renglones las ventas de 2025 y acumula el número de renglones y el tamaño del lote más grande. En memoria sólo hay un lote a la vez.

In [ ]:
total = 0
lote_mayor = 0
for lote in pl.scan_parquet(dir_por_anio_pb).filter(pl.col('anio') == 2025).collect_batches(chunk_size=100_000):
    total += lote.height
    lote_mayor = max(lote_mayor, lote.height)

print(f"Renglones de 2025: {total:,} · lote más grande: {lote_mayor:,}")

### Ejercicio.

Escribe las ventas grandes (`ventas_grandes`) particionadas por `categoria` en `data_tmp/ventas_grandes_por_categoria_38` con `pl.PartitionBy`. Después, recorre por lotes todo el conjunto con `collect_batches()` y obtén `resumen_categorias`: una tupla `(numero_de_carpetas, total_de_renglones)`.

* La siguiente celda es para tu solución: sustituye `None` por el código que calcula `resumen_categorias`.

In [ ]:
resumen_categorias = None  # Sustituye None por tu solución.

**Comprueba tu resultado.**

* La siguiente celda comprueba tu resultado con `assert`. Si algo no coincide, muestra un mensaje que indica qué revisar; si todavía no resuelves el ejercicio, sólo lo indica.

In [ ]:
def comprobar_categorias(resultado):
    carpeta = os.path.join(dir_tmp, 'ventas_grandes_por_categoria_38')
    assert os.path.isdir(carpeta), 'No existe data_tmp/ventas_grandes_por_categoria_38; revisa la carpeta de pl.PartitionBy.'
    assert isinstance(resultado, tuple) and len(resultado) == 2, 'El resultado debe ser una tupla con dos valores.'
    carpetas, renglones = resultado
    assert carpetas == len(CATEGORIAS), f'Debe haber una carpeta por categoría ({len(CATEGORIAS)}); se contaron {carpetas}.'
    assert renglones == N_RENGLONES, f'El total de renglones debe ser {N_RENGLONES:,}; se obtuvo {renglones:,}.'
    print('Correcto.')


if resumen_categorias is None:
    print('Ejercicio pendiente: completa la celda anterior y vuelve a ejecutar esta.')
else:
    comprobar_categorias(resumen_categorias)

**Solución.**

* La siguiente celda contiene una solución posible y la comprueba con la misma función. Su código está oculto: intenta resolver el ejercicio antes de mostrarlo (haz clic en la barra a la izquierda de la celda).

In [ ]:
dir_por_categoria = os.path.join(dir_tmp, 'ventas_grandes_por_categoria_38')
shutil.rmtree(dir_por_categoria, ignore_errors=True)
ventas_grandes.drop('anio').sink_parquet(pl.PartitionBy(dir_por_categoria, key='categoria'), mkdir=True)

numero_de_carpetas = len(os.listdir(dir_por_categoria))
total_de_renglones = sum(lote.height for lote in pl.scan_parquet(dir_por_categoria).collect_batches())
resumen_categorias = (numero_de_carpetas, total_de_renglones)

comprobar_categorias(resumen_categorias)

## 📘 Autoestudio: Medición de rendimiento: *pandas* y *Polars*.

> **📘 Autoestudio.** Esta sección no se ve en clase y ningún otro *notebook* ni sección de clase depende de ella.
>
> * **Qué aprenderás:** comparar de forma justa el tiempo de una misma operación en *pandas* y en *Polars*, en modo inmediato y diferido.
> * **Requisitos:** las secciones de clase de este *notebook* y la agrupación con *pandas* del *notebook* 24.
> * **Tiempo estimado:** 20 minutos.
> * **Cómo saber si lo hiciste bien:** al final hay un ejercicio y una celda que comprueba tu resultado.

Comparar bibliotecas requiere cuidado para que la medición sea justa:

* **Mismos datos y misma operación.** Se generan una sola vez y ambos *dataframes* se construyen **antes** de medir, porque lo que interesa es el tiempo de la operación, no el de la carga.
* **Varias repeticiones.** La primera ejecución puede ser más lenta, así que se toma el menor de varios tiempos con la función `mejor_tiempo()` del ejemplo completo.
* **Resultados que dependen del equipo.** *Polars* usa todos los núcleos del procesador y *pandas* casi siempre uno solo, así que la diferencia cambia según la máquina. En *GitHub Codespaces*, la máquina más pequeña tiene 2 núcleos.

La operación es típica de un análisis: filtrar, agrupar por una columna de texto y calcular dos estadísticas. Se usan `N_RENGLONES` renglones, el mismo tamaño del ejemplo completo.

* La siguiente celda genera los datos con *NumPy* y construye ambos *dataframes*. Esta parte no se mide.

In [ ]:
import pandas as pd

rng_medicion = np.random.default_rng(37)
datos = {
    'tienda': rng_medicion.choice([f'T{i:02d}' for i in range(1, 51)], N_RENGLONES),
    'cantidad': rng_medicion.integers(1, 6, N_RENGLONES),
    'precio': rng_medicion.uniform(50, 2000, N_RENGLONES).round(2),
}

df_pandas = pd.DataFrame(datos)
df_polars = pl.DataFrame(datos)
print(f"{N_RENGLONES:,} renglones · {os.cpu_count()} núcleos disponibles")

* La siguiente celda define las tres versiones de la operación: *pandas*, *Polars* en modo inmediato y *Polars* en modo diferido.

In [ ]:
def con_pandas():
    filtrado = df_pandas[df_pandas['cantidad'] >= 2]
    return filtrado.assign(importe=filtrado['cantidad'] * filtrado['precio']).groupby('tienda').agg(
        importe_total=('importe', 'sum'), precio_medio=('precio', 'mean')
    )


def con_polars():
    return (
        df_polars
        .filter(pl.col('cantidad') >= 2)
        .group_by('tienda')
        .agg(
            (pl.col('cantidad') * pl.col('precio')).sum().alias('importe_total'),
            pl.col('precio').mean().alias('precio_medio'),
        )
    )


def con_polars_lazy():
    return (
        df_polars.lazy()
        .filter(pl.col('cantidad') >= 2)
        .group_by('tienda')
        .agg(
            (pl.col('cantidad') * pl.col('precio')).sum().alias('importe_total'),
            pl.col('precio').mean().alias('precio_medio'),
        )
        .collect()
    )

* La siguiente celda verifica que las tres versiones dan el mismo resultado antes de medir. Se compara con `np.allclose()` porque las sumas de punto flotante pueden diferir en los últimos decimales según el orden en que se suman.

In [ ]:
referencia = con_polars().sort('tienda')
resultados = {
    'pandas': pl.from_pandas(con_pandas().reset_index()).sort('tienda'),
    'polars diferido': con_polars_lazy().sort('tienda'),
}
for nombre, resultado_i in resultados.items():
    iguales = all(np.allclose(referencia[columna], resultado_i[columna]) for columna in ['importe_total', 'precio_medio'])
    print(f"{nombre}: {iguales}")

* La siguiente celda mide las tres versiones. Los tiempos y la proporción varían de un equipo a otro.

In [ ]:
t_pandas = mejor_tiempo(con_pandas)
t_polars = mejor_tiempo(con_polars)
t_lazy = mejor_tiempo(con_polars_lazy)

print(f"pandas:                  {t_pandas:.3f} s")
print(f"Polars (inmediato):      {t_polars:.3f} s  ({t_pandas / t_polars:.1f} veces más rápido)")
print(f"Polars (diferido):       {t_lazy:.3f} s  ({t_pandas / t_lazy:.1f} veces más rápido)")

La diferencia proviene de varias fuentes: *Polars* reparte el filtro y la agrupación entre los núcleos disponibles, evita crear columnas intermedias (el importe se calcula dentro de la agregación) y trabaja directamente sobre el formato de *Arrow*. Con pocos datos o con un solo núcleo, la diferencia se reduce.

### Ejercicio.

Escribe una cuarta versión de la operación: la consulta diferida de `con_polars_lazy()`, pero ejecutada con el motor en memoria (`engine='in-memory'`). Obtén `resultado_en_memoria`: su resultado ordenado por `tienda`. Si quieres, mide también su tiempo con `mejor_tiempo()`.

* La siguiente celda es para tu solución: sustituye `None` por el código que calcula `resultado_en_memoria`.

In [ ]:
resultado_en_memoria = None  # Sustituye None por tu solución.

**Comprueba tu resultado.**

* La siguiente celda comprueba tu resultado con `assert`. Si algo no coincide, muestra un mensaje que indica qué revisar; si todavía no resuelves el ejercicio, sólo lo indica.

In [ ]:
def comprobar_en_memoria(resultado):
    assert isinstance(resultado, pl.DataFrame), 'El resultado debe ser un DataFrame de Polars.'
    assert resultado['tienda'].to_list() == referencia['tienda'].to_list(), 'Las tiendas no coinciden o no están ordenadas.'
    for columna in ['importe_total', 'precio_medio']:
        assert np.allclose(resultado[columna], referencia[columna]), f'La columna {columna} no coincide con la referencia.'
    print('Correcto.')


if resultado_en_memoria is None:
    print('Ejercicio pendiente: completa la celda anterior y vuelve a ejecutar esta.')
else:
    comprobar_en_memoria(resultado_en_memoria)

**Solución.**

* La siguiente celda contiene una solución posible y la comprueba con la misma función. Su código está oculto: intenta resolver el ejercicio antes de mostrarlo (haz clic en la barra a la izquierda de la celda).

In [ ]:
resultado_en_memoria = (
    df_polars.lazy()
    .filter(pl.col('cantidad') >= 2)
    .group_by('tienda')
    .agg(
        (pl.col('cantidad') * pl.col('precio')).sum().alias('importe_total'),
        pl.col('precio').mean().alias('precio_medio'),
    )
    .collect(engine='in-memory')
    .sort('tienda')
)

comprobar_en_memoria(resultado_en_memoria)

## Limpieza.

* La siguiente celda borra las carpetas de datos grandes que creó este *notebook* en `data_tmp/`, para liberar espacio en disco. Si quieres repetir alguna sección, vuelve a ejecutar el *notebook* desde el principio.

In [ ]:
for carpeta in ['ventas_grandes_38', 'ventas_grandes_por_anio_38', 'ventas_grandes_por_anio_pb_38', 'ventas_grandes_por_categoria_38']:
    shutil.rmtree(os.path.join(dir_tmp, carpeta), ignore_errors=True)
print(sorted(os.listdir(dir_tmp)))

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| **Modo diferido** | `df.lazy()` y `scan_*()` crean un `LazyFrame`: un plan de consulta que sólo se ejecuta con `collect()` o `sink_*()`. |
| **`collect_schema()`** | Obtiene el *schema* del resultado sin ejecutar la consulta. |
| **`scan_csv()` y `scan_parquet()`** | Lectura diferida; aceptan patrones como `parte_*.csv`, `n_rows` para muestras y, en *Parquet*, carpetas con el esquema de *Hive*. |
| **`explain()`** | Muestra el plan (de abajo hacia arriba), optimizado o tal como se escribió, en texto o en diagrama. |
| **Optimizaciones** | *Predicate*, *projection* y *slice pushdown*: se lee y procesa sólo lo necesario. |
| **`collect(engine=...)`** | En *Polars* 2.x, `'auto'` usa el motor *streaming* por omisión; `'in-memory'` carga cada paso completo en memoria. |
| **`sink_parquet()` y `sink_csv()`** | Escriben el resultado conforme se calcula, sin reunirlo en memoria. |
| **Esquema de *Hive*** | Carpetas `columna=valor`; `write_parquet(partition_by=...)` lo escribe y un filtro sobre la columna de partición sólo abre las carpetas necesarias. |
| **📘 Autoestudio** | *SQL* (`sql()`, `pl.SQLContext`), `pl.collect_all()`, `pl.PartitionBy` y `collect_batches()`, y la medición de rendimiento entre *pandas* y *Polars*. |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>